In [29]:
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error,confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA
from scipy import stats
import sys
import seaborn as sns
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *

In [30]:
import chipwhisperer as cw

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_1_1'
%run "/home/40265864@ecit.qub.ac.uk/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:397) Could not adjust adc_mul via output divider alone. Recalcing clocks...
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:398) Target clock has dropped for a moment. You may need to reset your target


scope.gain.gain                          changed from 20                        to 22                       
scope.gain.db                            changed from 24.174311926605505        to 25.091743119266056       
scope.adc.samples                        changed from 10000                     to 5000                     
scope.clock.clkgen_freq                  changed from 7692307.692307692         to 7370129.87012987         
scope.clock.adc_mul                      changed from 26                        to 4                        
scope.clock.adc_freq                     changed from 200000000.0               to 29480519.48051948        
scope.clock.extclk_tolerance             changed from 13096723.705530167        to 149880108.32439613       
scope.glitch.phase_shift_steps           changed from 4592                      to 4368                     
scope.ADS4128.low_speed                  changed from False                     to True                     


In [32]:
scope.adc.samples = 10000#12000                 # Number of samples per segment
scope.clock.adc_mul = 26
scope.gain.mode = "high"
scope.gain.gain = 20
scope.clock.reset_adc()

In [40]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../hardware/firmware/floating_point_multiply_test
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
.
.
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Welcome to another exciting ChipWhisperer target build!!
Compiling:
+--------------------------------------------------------
-en     floating_point_multiply_test.c ...
+ Built for platform Microchip SAM4S with:
+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = 
+--------------------------------------------------------
-e Done!
.
LINKING:
-en     floating_point_multiply_test-CW308_SAM4S.elf ...
-e Done!
.
.
.
.
Creating load file for Flash: floating_point_multiply_test-CW308_SAM4S.hex
Creating load file for Flash: floating_point_multiply_test-CW308_SAM4S.bin
Creating load file for EEPROM: floating_point_multiply_test-CW308_SAM4S.eep
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fu

In [41]:
cw.program_target(scope, prog, "../hardware/firmware/floating_point_multiply_test/floating_point_multiply_test-{}.hex".format(PLATFORM))

In [42]:
def gather_trace(a,b):
    time.sleep(0.05)
    scope.arm()
    initial_trig_count = scope.adc.trig_count

    target.write(f"{a}\n".encode("ascii"))

    time.sleep(0.05)
    target.write(f"{b}\n".encode("ascii"))

    line = target.read()
    print(line)

    if scope.capture():
        raise RuntimeError("Capture failed")

    trace = scope.get_last_trace()
    active_trig_count = scope.adc.trig_count# - initial_trig_count
    print(active_trig_count)
    return trace



In [43]:

def float_to_hex(f: float) -> str:
    # Pack Python float -> 32-bit IEEE754 single precision
    b = struct.pack('<f', f)       # little-endian, 4 bytes
    u = struct.unpack('<I', b)[0]  # interpret as unsigned int
    return f"0x{u:08X}"

In [44]:
traces_const = []
traces_rnd = []

repeats = 1000

a = 0.13252
b = 1.23534

##warmup
for _ in range(10):
    gather_trace(a,b)

print("\nWARMUP DONE\n")

for _ in range(repeats):
    traces_const.append(gather_trace(a,b))

for _ in range(repeats):
    c = np.random.random()
    d = np.random.random()
    traces_rnd.append(gather_trace(c,d))

FP_MUL_READY

Give two floats: 

8918
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3EB28DF2
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3EA80348
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3DC7BF85
Give two floats: 

6630
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3E04FE18
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3E92F3DF
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3CE6F427
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3E8FF1AE
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3DAAB4F9
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3DBE8C0A
Give two floats: 

5850

WARMUP DONE

Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3EEF9743
Give two floats: 

5850
Input a: 0x3E07B353
Input b: 0x3F9E1F9F
Outout c: 0x3E94D67C
Give two floats: 

5850
Input a: 0x3E

KeyboardInterrupt: 

In [ ]:
const_group = np.asarray(traces_const)
rnd_group = np.asarray(traces_rnd)
np.save("const_masked_1_order.npy", const_group)
np.save("rnd_masked_1_order.npy", rnd_group)


In [ ]:
const_plain = np.load("const_simple.npy")

In [ ]:
cw.plot(const_plain[10]) * cw.plot(const_group[10])

In [ ]:
mean_const = np.mean(const_group, axis=0)
mean_rnd = np.mean(rnd_group, axis=0)

In [ ]:
cw.plot(mean_const) * cw.plot(mean_rnd)

In [ ]:
cw.plot(mean_const - mean_rnd)

In [ ]:
from scipy.stats import ttest_ind


t_val = ttest_ind(const_group, rnd_group, axis=0, equal_var=False)[0]
cv = cw.plot(t_val)
cv *= cw.plot([4.5]*len(const_group[0]))
cv *= cw.plot([-4.5]*len(const_group[0]))
cv

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')

lw=0.75
col1='g'
col2='r'
lst = "-"

fig, axs = plt.subplots()
axs.plot(t_val, linewidth=lw,color=col1)
#axs[0].plot(t_val_mcu[0])
axs.axhline(y=4.5, color=col2, linestyle=lst)
axs.axhline(y=-4.5, color=col2, linestyle=lst)
axs.set_xlabel("Sample Point",fontsize=12)
axs.set_ylabel("T-score",fontsize=12)
axs.set_title("ARM Cortex-M4 Floating Point Multiply",fontsize=12)

plt.tight_layout()
plt.savefig("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/results/M4_fp_mult_tvla_simple.png",dpi=500)



In [ ]:
N = len(const_group)
t_val = ttest_ind(rnd_group[N//2:], rnd_group[:N//2], axis=0, equal_var=False)[0]
cv = cw.plot(t_val)
cv *= cw.plot([4.5]*len(const_group[0]))
cv *= cw.plot([-4.5]*len(const_group[0]))
cv